<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.es/cap04/cap04.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Práctica con Ejercicios de Programación**


Esta lista transforma los conceptos del Capítulo 4 en una ruta práctica de segmentación y morfología matemática. Los EPs comienzan con umbralización y avanzan hasta etiquetado y descriptores de componentes, siempre con matrices pequeñas para que cada píxel pueda verificarse a mano.

> ### ❗ Regla común de los EPs morfológicos
>
> En las operaciones con vecindad, **no haga padding**. Para cada píxel, evalúe únicamente las posiciones del elemento estructurante que caen dentro del dominio de la imagen. Esta es la misma idea de las implementaciones didácticas en `morph.py`, como `mm::dil0`, `mm::ero0`, `mm::dil1` y `mm::label0`: la vecindad se recorta por el dominio válido de la imagen.

### 🎯 Objetivo de este cuaderno

El cuaderno permite desarrollar, validar, organizar y probar soluciones de **Ejercicios de Programación (EPs)** en entornos interactivos, como Colab, con los mismos casos de prueba de Moodle, copiándolos allí solo al momento de registrar la nota oficial.

#### *Download*

Descargue `morph.py` y `testsuite.py` ejecutando la celda a continuación:

In [ ]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatos de build del trayecto C++ (.cpp, binario, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# El kernel es Python incluso en el trayecto C++: `mm` (morph.py) es usado por los
# simuladores, por la exhibición de las figuras que el binario C++ genera y por el
# estado mm::Image entre celdas. cpp=True descarga también el trayecto compilado
# (morph.hpp + stb_image*.h), usado en el #include de las celdas %%writefile *.cpp.
import config
config.setup(testsuite=True, cpp=True)
from morph import mm
from testsuite import TestSuite

#### Ejecutando las pruebas
Para evaluar las pruebas, ejecute `TestSuite("EP04_01.extensión").run()` en una nueva celda, reemplazando la extensión por la del lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema descarga los casos de prueba de GitHub, ejecuta el programa y calcula la nota automáticamente.

Para probar código Python directamente, sin guardar archivo, use `run_code(codigo)` pasando el código como *string* en una variable `codigo`:

```python
codigo = """
from morph import mm
# ... su código aquí ...
"""
TestSuite("EP04_01").run_code(codigo)
```

### EP04_01 🎚️ Umbralización Global por Umbral Fijo

En los **escáneres de documentos** y en los **sistemas de lectura de códigos de barras**, la primera etapa del procesamiento siempre consiste en separar lo que es "objeto" (tinta, texto, barras) de lo que es "fondo" (papel, embalaje). La **umbralización global** hace exactamente esto: compara cada píxel con un único umbral $T$ y decide, en tiempo real, si pertenece a la clase clara o a la clase oscura. Es el operador de segmentación más simple — y aun así, está detrás de buena parte de los *pipelines* industriales de inspección visual.
Ver en [Figura 4.1](#fig-04-sim-ep0401-limiar) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Umbral:** Leer el entero $T$ (umbral de decisión).
3. **Datos:** Leer los valores enteros de la matriz original fila a fila.
4. **Mapeo:** Para cada píxel $p$, calcular el nuevo valor mediante la ecuación:

$$
p' =
\begin{cases}
255, & \text{si } p > T \\
0, & \text{si } p \le T
\end{cases}
$$
5. **Salida:** Mostrar la matriz binarizada con dimensiones $L \times C$.

#### 📌 Restricciones Computacionales

* **Binarización:** La salida contiene **solo** los valores $0$ o $255$.
* **Comparación estricta:** El criterio usa $> T$ (los píxeles iguales a $T$ se convierten en fondo).
* **Tipo:** El resultado final debe ser entero.
* **Observación:** Este EP sigue la convención de OpenCV (`cv2.THRESL_BINARY`): solo los píxeles con valor **mayor que** $T$ se convierten en blancos (`255`); los píxeles con valor **igual a** $T$ permanecen negros (`0`).

#### 🧠 Fundamentación Teórica

| Parámetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$T$ pequeño** | Entero | La mayoría de los píxeles se vuelven blancos |
| **$T$ grande**  | Entero | La mayoría de los píxeles se vuelven negros |
| **$T$ bien elegido** | Entero | Separa nítidamente objeto y fondo |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $T$.
* Líneas siguientes: Elementos enteros de la matriz original.

**Salida:**

* Matriz binarizada en $L$ filas y $C$ columnas, valores $0$ o $255$ separados por espacio.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 2<br>4<br>100<br>0 99 100 180<br>255 30 120 80 | 0 0 0 255<br>255 0 255 0 | $T=100$: solo los píxeles con valor mayor que 100 se vuelven blancos; <br>por eso, 99 y 100 se vuelven negros. |
| 1<br>3<br>0<br>0 50 255 | 0 255 255 | $T=0$: solo los píxeles con valor estrictamente mayor que 0 se vuelven blancos. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0401-limiar" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎚️ Simulador EP04_01: Umbralización Global</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = (p > T) ? 255 : 0</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Haga clic en una celda de <b>Entrada Original</b> para oscurecer el píxel (−30) y haga clic con el botón derecho para aclarar (+30). Ajuste el umbral T para la binarización.</p>

    <!-- Controle do Limiar T -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">T (Umbral)</label>
        <span id="sim_ep0401_vl_t" style="font-family:monospace;font-size:12px;font-weight:700;color:#2980b9;">128</span>
      </div>
      <input type="range" id="sim_ep0401_sl_t" min="0" max="255" step="1" value="128" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado Binarizado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (Clicable)</span>
        <div id="sim_ep0401_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Resultado Binarizado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Binarizado (p')</span>
        <div id="sim_ep0401_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Restablecer Umbral (T = 128)</button>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0401_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula aplicada: <b>(p > 128) ? 255 : 0</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0401(root){
    if (!root || root.dataset.simEp0401Init) return;
    root.dataset.simEp0401Init = "1";

    var slT      = root.querySelector('#sim_ep0401_sl_t');
    var vlT      = root.querySelector('#sim_ep0401_vl_t');
    var gridOrig = root.querySelector('#sim_ep0401_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0401_grid_new');
    var debugDiv = root.querySelector('#sim_ep0401_debug');

    var btnNew   = root.querySelector('#sim_ep0401_btnNew');
    var btnReset = root.querySelector('#sim_ep0401_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-30) | Botão direito: clareia (+30)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 30);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 30);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function render() {
      var T = parseInt(slT.value, 10) || 0;
      vlT.textContent = T;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>(p > ' + T + ') ? 255 : 0</b>';

      renderOrig();
      gridNew.innerHTML = '';

      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slT.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slT.value = '128';
      render();
    });

    render();
  }

  function tryInitSimEP0401(){
    var root = document.getElementById('sim-ep0401-limiar');
    if (root) initSimEP0401(root); else setTimeout(tryInitSimEP0401, 200);
  }
  tryInitSimEP0401();
})();
</script>
</div>
""")

**Figura 4.1:** Simulador EP04_01: Umbralización Global por Umbral Fijo (p


<figure id="fig-04-sim-ep0401-limiar">
  <img src="imagens/fig-04-sim-ep0401-limiar.png" alt=" Simulador EP04_01: Umbralización Global por Umbral Fijo (p' = (p > T) ? 255 : 0) " style="max-width:80%" />
  <figcaption><strong>Figura 4.1:</strong>  Simulador EP04_01: Umbralización Global por Umbral Fijo (p' = (p > T) ? 255 : 0) </figcaption>
</figure>

In [ ]:
%%writefile EP04_01.cpp
// your solution

In [ ]:
TestSuite("EP04_01.cpp").run()

### EP04_02 📊 Umbralización Automática de Otsu

Elegir manualmente el umbral $T$ funciona cuando la iluminación es estable, pero en **microscopía digital** y en **inspección de láminas de sangre**, cada muestra tiene un contraste diferente — un umbral fijo fallaría de imagen en imagen. El **método de Otsu** resuelve esto encontrando, por sí solo, el umbral que **maximiza la separación estadística** entre las dos clases de píxeles, haciendo la segmentación automática y adaptativa.
Ver en [Figura 4.2](#fig-04-sim-ep0402-otsu) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer los valores enteros de la matriz original fila por fila.
3. **Histograma:** Construir el histograma $h[i]$, $i=0,\dots,255$, contando cuántos píxeles tienen valor $i$.
4. **Búsqueda del umbral:** Para cada candidato $T$ de $1$ a $255$, calcular la **varianza entre clases**:
$$
\sigma_B^2(T) = \frac{n_0 \cdot n_1}{N^2}\,(m_0 - m_1)^2
$$
donde $n_0,n_1$ son las cantidades de píxeles con valor $<T$ y $\geq T$, $m_0,m_1$ son sus medias, y $N=L\times C$.

5. **Elección:** El umbral óptimo $T^*$ es el que maximiza $\sigma_B^2(T)$ (en caso de empate, mantener el **primero** encontrado).
6. **Aplicación:** Binarizar la imagen usando $T^*$, aplicando:
$$
p' =
\begin{cases}
255, & \text{si } p > T^* \\
0, & \text{si } p \le T^*
\end{cases}
$$

#### 📌 Restricciones Computacionales

* **Candidatos válidos:** Ignorar $T$ que deje $n_0=0$ o $n_1=0$ (clase vacía).
* **Empate:** Mantener siempre el **primer** $T$ que alcanzó el valor máximo de $\sigma_B^2$.
* **Tipo:** $T^*$ y la matriz de salida deben ser enteros.
* **Convención OpenCV:** La binarización sigue `cv2.THRESL_BINARY`; los píxeles con valor exactamente igual a $T^*$ se vuelven negros.

#### 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto |
|----------|-------------|---------|
| **$\sigma_B^2(T)$ alta** | Clases bien separadas en $T$ | $T$ es un buen candidato a umbral |
| **Histograma bimodal** | Dos "picos" distintos | Otsu encuentra el valle entre ellos |
| **Histograma unimodal** | Un único "pico" | Otsu aún elige *algún* $T$, pero la segmentación es poco fiable |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos enteros de la matriz original.

**Salida:**

* Matriz binarizada en $L$ filas y $C$ columnas, valores $0$ o $255$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 4<br>4<br>12 12 12 200<br>12 12 200 200<br>12 200 200 200<br>200 200 200 200 | 0 0 0 255<br>0 0 255 255<br>0 255 255 255<br>255 255 255 255 | Histograma bimodal claro: 12 y 200 |
| 1<br>2<br>10 250 | 0 250 | Solo dos valores: $T^*$ queda en el mayor |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0402-otsu" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 Simulador EP04_02: Otsu Automático</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">T* = argmax σ²_B(T)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Clic izquierdo oscurece (−25) y clic con el botón derecho aclara (+25) los píxeles de la entrada. Observe el umbral óptimo T* ajustarse dinámicamente al histograma.</p>

    <!-- Painel do Histograma e T* -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;">
      <div id="sim_ep0402_hist" style="display:flex;align-items:flex-end;gap:2px;height:100px;margin-bottom:8px;border-bottom:1px solid #e4dcc8;padding-bottom:2px;"></div>
      <p id="sim_ep0402_info" style="text-align:center;font-size:11.5px;font-family:monospace;font-weight:700;color:#26241d;margin:0;">T* = −</p>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Clicável vs Resultado Otsu -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (Clicable)</span>
        <div id="sim_ep0402_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Otsu -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Otsu (p')</span>
        <div id="sim_ep0402_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botão de Nova Imagem -->
    <div style="text-align:center;">
      <button id="sim_ep0402_btnNew" style="padding:6px 14px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen (Dos Grupos)</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0402(root){
    if (!root || root.dataset.simEp0402Init) return;
    root.dataset.simEp0402Init = "1";

    var gridOrig = root.querySelector('#sim_ep0402_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0402_grid_new');
    var info     = root.querySelector('#sim_ep0402_info');
    var hist     = root.querySelector('#sim_ep0402_hist');
    var btnNew   = root.querySelector('#sim_ep0402_btnNew');

    var pixels = [];

    function generate() {
      var c1 = 20 + Math.floor(Math.random() * 40);
      var c2 = 180 + Math.floor(Math.random() * 60);
      pixels = [];
      for (var i = 0; i < 16; i++) {
        var base = (Math.random() < 0.5) ? c1 : c2;
        pixels.push(Math.max(0, Math.min(255, base + Math.floor(Math.random() * 16 - 8))));
      }
    }

    function otsu(pix) {
      var histArr = new Array(256).fill(0);
      pix.forEach(function(p){ histArr[p]++; });
      var N = pix.length, bestVar = -1, bestT = 0;
      var total = pix.reduce(function(a, b){ return a + b; }, 0);

      for (var T = 1; T < 256; T++) {
        var n0 = 0, s0 = 0;
        for (var i = 0; i < T; i++) {
          n0 += histArr[i];
          s0 += i * histArr[i];
        }
        var n1 = N - n0, s1 = total - s0;
        if (n0 === 0 || n1 === 0) continue;
        var m0 = s0 / n0, m1 = s1 / n1;
        var v = (n0 * n1) * (m0 - m1) * (m0 - m1) / (N * N);
        if (v > bestVar) {
          bestVar = v;
          bestT = T;
        }
      }
      return bestT;
    }

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-25) | Botão direito: clareia (+25)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 25);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 25);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function renderHist(T) {
      hist.innerHTML = '';
      var histArr = new Array(256).fill(0);
      pixels.forEach(function(p){ histArr[p]++; });
      var maxH = Math.max.apply(null, histArr);

      for (var i = 0; i < 256; i += 4) {
        var h = (histArr[i] / (maxH || 1)) * 100;
        var bar = document.createElement('div');
        var col = (i >= T) ? '#2980b9' : '#8a8371';
        bar.style.cssText = 'flex:1;height:' + h + '%;background:' + col + ';border-radius:2px 2px 0 0;';
        hist.appendChild(bar);
      }
    }

    function render() {
      var T = otsu(pixels);
      info.innerHTML = 'T* encontrado = <b>' + T + '</b>';
      renderOrig();
      renderHist(T);

      gridNew.innerHTML = '';
      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0402(){
    var root = document.getElementById('sim-ep0402-otsu');
    if (root) initSimEP0402(root); else setTimeout(tryInitSimEP0402, 200);
  }
  tryInitSimEP0402();
})();
</script>
</div>
""")

**Figura 4.2:** Simulador EP04_02: Limiarización Automática de Otsu (T* = argmax σ²_B(T))


<figure id="fig-04-sim-ep0402-otsu">
  <img src="imagens/fig-04-sim-ep0402-otsu.png" alt=" Simulador EP04_02: Limiarización Automática de Otsu (T* = argmax σ²_B(T)) " style="max-width:80%" />
  <figcaption><strong>Figura 4.2:</strong>  Simulador EP04_02: Limiarización Automática de Otsu (T* = argmax σ²_B(T)) </figcaption>
</figure>

In [ ]:
%%writefile EP04_02.cpp
// your solution

In [ ]:
TestSuite("EP04_02.cpp").run()

### EP04_03 🌱 Dilatación Binaria Plana (mm.dil0)

En **microscopía de partículas** y en **OCR de placas de automóvil desgastadas**, los trazos finos o discontinuos deben "engrosarse" para que el reconocimiento funcione. La **dilatación morfológica** hace exactamente eso: expande regiones claras usando un elemento estructurante $B$ — la misma operación implementada en `morph.py` como `mm::dil0(f, B)`, usada cuando $B$ es **plano** (sin pesos, solo $0$/$1$).
Ver en [Figura 4.3](#fig-04-sim-ep0403-dilatacao) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, fila a fila.
4. **Datos:** Leer la matriz $f$ (la imagen original), fila a fila.
5. **Reflexión:** Construir $B_{ref}$, la versión de $B$ reflejada en $180°$ (filas y columnas invertidas) — exactamente como hace `mm::dil0` internamente.
6. **Vecindad sin padding:** Para cada píxel $(y,x)$, recorrer las posiciones $(by,bx)$ de $B_{ref}$ centradas en $(y,x)$, usando el desplazamiento
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Descartar** todo $(v_y,v_x)$ fuera de $[0,L)\times[0,C)$ — **no rellenar con ceros**.
7. **Mapeo:** Calcular cada píxel de salida como el **máximo** entre $f(y,x)$ y todos los $f(v_y,v_x)$ válidos cuya posición correspondiente en $B_{ref}$ vale $1$:
$$
g(y,x) = \max\Big(f(y,x),\ \max_{\substack{(v_y,v_x)\ \text{válido}\\ B_{ref}(by,bx)=1}} f(v_y,v_x)\Big)
$$
8. **Salida:** Mostrar la matriz $g$ con dimensiones $L \times C$.

#### 📌 Restricciones Computacionales

* **Sin padding:** Nunca inventar vecinos fuera de la imagen; usar solo los que existen realmente.
* **Reflexión obligatoria:** $B$ debe reflejarse antes de aplicarse (es lo que diferencia `mm::dil0` de una simple búsqueda de máximo).
* **Robustez de borde:** Si ninguna posición válida de $B_{ref}=1$ cae dentro del dominio para un píxel dado, este **mantiene su valor original**.

#### 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Dilatación** | $g \geq f$ siempre (extensiva) | Las regiones claras crecen, los huecos oscuros se encogen |
| **$B$ mayor** | Vecindad más amplia | Crecimiento más agresivo |
| **Reflexión de $B$** | $B_{ref}(y,x) = B(-y,-x)$ | Garantiza la definición formal de Minkowski de la dilatación |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros de la matriz $f$.

**Salida:**

* Matriz $g$ en $L$ filas y $C$ columnas, valores enteros separados por espacio.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0<br>0 9 0<br>0 0 0 | 0 9 0<br>9 9 9<br>0 9 0 | $B$ en cruz simétrico: punto aislado se expande en cruz |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 200 200 200 80 | $B$ horizontal: cada píxel "atrae" el máximo de los vecinos de la fila |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0403-dilatacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌱 Simulador EP04_03: Dilatación Plana (mm.dil0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Cambie el elemento estructurante B (o seleccione los preajustes) y haga clic en las celdas de la imagen original f para encender o apagar píxeles.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Elemento Estructurante B (Clic para Alternar 0/1)</span>
      <div id="sim_ep0403_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0403_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Cruz</button>
        <button id="sim_ep0403_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Cuadro</button>
        <button id="sim_ep0403_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonal</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Dilatada g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen Original f (5×5)</span>
        <div id="sim_ep0403_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0403_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Imagem Dilatada g -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Dilatada g (f ⊕ B)</span>
        <div id="sim_ep0403_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0403_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = máximo sobre vecinos válidos de B reflejado
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0403(root){
    if (!root || root.dataset.simEp0403Init) return;
    root.dataset.simEp0403Init = "1";

    var gB      = root.querySelector('#sim_ep0403_grid_B');
    var gO      = root.querySelector('#sim_ep0403_grid_orig');
    var gN      = root.querySelector('#sim_ep0403_grid_new');
    var debugDiv= root.querySelector('#sim_ep0403_debug');

    var btnNew   = root.querySelector('#sim_ep0403_btnNew');
    var btnCross = root.querySelector('#sim_ep0403_btnCross');
    var btnBox   = root.querySelector('#sim_ep0403_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0403_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 0 : 1);
        }
        pixels.push(row);
      }
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) {
        out.push(M[i].slice().reverse());
      }
      return out;
    }

    function dilate(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var Bref = reflect(Bm);
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] > g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#16a085' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = dilate(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0403(){
    var root = document.getElementById('sim-ep0403-dilatacao');
    if (root) initSimEP0403(root); else setTimeout(tryInitSimEP0403, 200);
  }
  tryInitSimEP0403();
})();
</script>
</div>
""")

**Figura 4.3:** Simulador EP04_03: Dilatación Binaria Plana (g = f ⊕ B)


<figure id="fig-04-sim-ep0403-dilatacao">
  <img src="imagens/fig-04-sim-ep0403-dilatacao.png" alt=" Simulador EP04_03: Dilatación Binaria Plana (g = f ⊕ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.3:</strong>  Simulador EP04_03: Dilatación Binaria Plana (g = f ⊕ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_03.cpp
// your solution

In [ ]:
TestSuite("EP04_03.cpp").run()

### EP04_04 🪨 Erosión Binaria Plana (mm.ero0)

Si la dilatación engrosa, la **erosión** afina. En **sistemas de conteo de células**, se utiliza para **separar células que se tocan**: al "comer" los bordes de cada región, las conexiones finas entre objetos desaparecen incluso antes de que se realice cualquier conteo. En `morph.py`, esta es la operación `mm::ero0(f, B)` — la **dual** exacta de la dilatación, y la única de las dos que **no** refleja el elemento estructurante.
Ver en [Figura 4.4](#fig-04-sim-ep0404-erosao) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, fila a fila.
4. **Datos:** Leer la matriz $f$ (la imagen original), fila a fila.
5. **Vecindad sin padding (¡sin reflexión!):** Para cada píxel $(y,x)$, recorrer las posiciones $(by,bx)$ de $B$ **en el orden original** (sin reflejar), usando el mismo desplazamiento del EP04_03:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$

**Descartar** todo $(v_y,v_x)$ fuera de $[0,L)\times[0,C)$.
6. **Mapeo:** Calcular cada píxel de salida como el **mínimo** entre $f(y,x)$ y todos los $f(v_y,v_x)$ válidos cuya posición correspondiente en $B$ vale $1$:
$$
g(y,x) = \min\Big(f(y,x),\ \min_{\substack{(v_y,v_x)\ \text{válido}\\ B(by,bx)=1}} f(v_y,v_x)\Big)
$$
7. **Salida:** Mostrar la matriz $g$ con dimensiones $L \times C$.

#### 📌 Restricciones Computacionales

* **Sin reflexión:** A diferencia de la dilatación, $B$ se usa **exactamente como se lee** — reflejarlo aquí sería un error conceptual grave.
* **Sin padding:** Los vecinos fuera de la imagen simplemente se ignoran, nunca se tratan como $0$.
* **Robustez de borde:** Si ninguna posición válida de $B=1$ cae dentro del dominio, el píxel mantiene su valor original.

#### 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Erosión** | $g \leq f$ siempre (anti-extensiva) | Las regiones claras se encogen, el ruido puntual desaparece |
| **Dualidad** | $\text{ero}(f,B) = -\text{dil}(-f, B_{ref})$ | La erosión y la dilatación son "espejos" matemáticos |
| **$B$ más grande** | Erosión más agresiva | Los objetos finos desaparecen por completo |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros de la matriz $f$.

**Salida:**

* Matriz $g$ en $L$ filas y $C$ columnas, valores enteros separados por espacios.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>9 9 9<br>9 0 9<br>9 9 9 | 9 0 9<br>0 0 0<br>9 0 9 | El "agujero" central (0) se propaga en cruz |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 10 5 5 80 | $B$ horizontal: cada píxel "extrae" el mínimo de los vecinos de la fila |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0404-erosao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪨 Simulador EP04_04: Erosión Plana (mm.ero0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Alterne el elemento estructurante B (o seleccione los presets) y haga clic en las celdas de la imagen original f para encender o apagar píxeles.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Elemento Estructurante B (Clic para Alternar 0/1)</span>
      <div id="sim_ep0404_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0404_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Cruz</button>
        <button id="sim_ep0404_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Caja</button>
        <button id="sim_ep0404_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonal</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Erodida g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen Original f (5×5)</span>
        <div id="sim_ep0404_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0404_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Imagem Erodida g -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Erosionada g (f ⊖ B)</span>
        <div id="sim_ep0404_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0404_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = mínimo sobre vecinos válidos de B (sin reflejar)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0404(root){
    if (!root || root.dataset.simEp0404Init) return;
    root.dataset.simEp0404Init = "1";

    var gB      = root.querySelector('#sim_ep0404_grid_B');
    var gO      = root.querySelector('#sim_ep0404_grid_orig');
    var gN      = root.querySelector('#sim_ep0404_grid_new');
    var debugDiv= root.querySelector('#sim_ep0404_debug');

    var btnNew   = root.querySelector('#sim_ep0404_btnNew');
    var btnCross = root.querySelector('#sim_ep0404_btnCross');
    var btnBox   = root.querySelector('#sim_ep0404_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0404_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 1 : 0);
        }
        pixels.push(row);
      }
    }

    function erode(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] < g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#c0392b' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = erode(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0404(){
    var root = document.getElementById('sim-ep0404-erosao');
    if (root) initSimEP0404(root); else setTimeout(tryInitSimEP0404, 200);
  }
  tryInitSimEP0404();
})();
</script>
</div>
""")

**Figura 4.4:** Simulador EP04_04: Erosión Binaria Plana (g = f ⊖ B)


<figure id="fig-04-sim-ep0404-erosao">
  <img src="imagens/fig-04-sim-ep0404-erosao.png" alt=" Simulador EP04_04: Erosión Binaria Plana (g = f ⊖ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.4:</strong>  Simulador EP04_04: Erosión Binaria Plana (g = f ⊖ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_04.cpp
// your solution

In [ ]:
TestSuite("EP04_04.cpp").run()

### EP04_05 🧹 Apertura Morfológica (Eliminación de Ruido)

Las imágenes capturadas por **sensores de bajo costo**, como los de drones agrícolas, suelen venir salpicadas de pequeños puntos de ruido — píxeles aislados que no representan nada real. Aplicar erosión seguida de dilatación con el **mismo** elemento estructurante produce la **apertura**: esta "limpia" puntos y protuberancias finas, pero devuelve al objeto principal prácticamente su tamaño original. Es la combinación clásica utilizada en **preprocesamiento de imágenes de satélite** antes de cualquier conteo de área plantada.
Ver en [Figura 4.5](#fig-04-sim-ep0405-abertura) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, fila a fila.
4. **Datos:** Leer la matriz binaria $f$ (valores $0$ o $1$), fila a fila.
5. **Erosión:** Calcular $e = f \ominus B$, usando exactamente el algoritmo del EP04_04 (sin reflejar $B$, sin padding).
6. **Dilatación:** Calcular $g = e \oplus B$, usando exactamente el algoritmo del EP04_03 (reflejando $B$, sin padding) — pero ahora aplicado sobre $e$, no sobre $f$.
7. **Salida:** Mostrar la matriz resultante $g$ (la **apertura** de $f$ por $B$) con dimensiones $L \times C$.

#### 📌 Restricciones Computacionales

* **Orden fijo:** Es **siempre** erosión primero, luego dilatación — el orden inverso define otro operador (cierre, del próximo EP).
* **Mismo $B$:** El elemento estructurante utilizado en la erosión y en la dilatación debe ser idéntico.
* **Sin padding en ninguna de las dos etapas.**

#### 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Antiextensividad** | $g \subseteq f$ siempre | La apertura nunca crea un píxel nuevo, solo elimina |
| **Idempotencia** | $\text{apertura}(\text{apertura}(f)) = \text{apertura}(f)$ | Aplicar de nuevo no cambia nada más |
| **Puntos aislados** | Menores que $B$ | Son completamente eliminados |
| **Núcleo del objeto** | Mayor que $B$ | Se recupera casi intacto mediante la dilatación final |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros ($0$ o $1$) de la matriz $f$.

**Salida:**

* Matriz resultante en $L$ filas y $C$ columnas, valores $0$ o $1$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 7<br>7<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0<br>0 1 0 0 0 1 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 1 0<br>0 0 0 0 0 0 0<br>0 1 0 0 0 0 1 | 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | Los puntos aislados y la protuberancia fina desaparecen; el cuadrado central sobrevive |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0405-abertura" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧹 Simulador EP04_05: Apertura Morfológica</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊖ B) ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en las celdas de <b>f original</b> para encender o apagar píxeles (¡crea tu propio ruido de fondo!) y ajusta el tamaño del elemento estructurante B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#8e44ad;">Tamaño de B (Caja n×n)</label><br>
      <input type="range" id="sim_ep0405_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#8e44ad;margin-top:6px;">
      <span id="sim_ep0405_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs e (Erosão) vs g (Abertura Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Clicable)</span>
        <div id="sim_ep0405_grid_f" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- e = f ⊖ B -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">e = f ⊖ B (Erosión)</span>
        <div id="sim_ep0405_grid_e" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = e ⊕ B -->
      <div style="background:#fafaf7;border:2px solid #8e44ad;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = e ⊕ B (Apertura)</span>
        <div id="sim_ep0405_grid_g" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0405_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen (Con Ruido)</button>
      <button id="sim_ep0405_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Limpiar Todo</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0405(root){
    if (!root || root.dataset.simEp0405Init) return;
    root.dataset.simEp0405Init = "1";

    var slN     = root.querySelector('#sim_ep0405_sl_n');
    var vlN     = root.querySelector('#sim_ep0405_vl_n');
    var gF      = root.querySelector('#sim_ep0405_grid_f');
    var gE      = root.querySelector('#sim_ep0405_grid_e');
    var gG      = root.querySelector('#sim_ep0405_grid_g');
    var btnNew  = root.querySelector('#sim_ep0405_btnNew');
    var btnClear= root.querySelector('#sim_ep0405_btnClear');

    var L = 7, C = 7, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 2; y < 5; y++) {
        for (var x = 2; x < 5; x++) f[y][x] = 1;
      }
      for (var k = 0; k < 3; k++) {
        var ry = Math.floor(Math.random() * L), rx = Math.floor(Math.random() * C);
        if (f[ry][rx] === 0 && (ry < 1 || ry > 5 || rx < 1 || rx > 5)) f[ry][rx] = 1;
      }
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#8e44ad' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#8e44ad' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var e = erode(f, B);
      var g = dilate(e, B);

      paintEditable(gF, f);
      paintStatic(gE, e);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0405(){
    var root = document.getElementById('sim-ep0405-abertura');
    if (root) initSimEP0405(root); else setTimeout(tryInitSimEP0405, 200);
  }
  tryInitSimEP0405();
})();
</script>
</div>
""")

**Figura 4.5:** Simulador EP04_05: Apertura Morfológica (g = (f ⊖ B) ⊕ B)


<figure id="fig-04-sim-ep0405-abertura">
  <img src="imagens/fig-04-sim-ep0405-abertura.png" alt=" Simulador EP04_05: Apertura Morfológica (g = (f ⊖ B) ⊕ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.5:</strong>  Simulador EP04_05: Apertura Morfológica (g = (f ⊖ B) ⊕ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_05.cpp
// your solution

In [ ]:
TestSuite("EP04_05.cpp").run()

### EP04_06 🧩 Cierre Morfológico (Relleno de Huecos)

En la **digitalización de huellas dactilares**, los surcos de la piel a veces se ven interrumpidos por suciedad o sequedad, creando pequeñas fallas en la curva continua que debería existir. El **cierre** —dilatación seguida de erosión con el mismo elemento estructurante— es el operador dual de la apertura: **rellena huecos pequeños y entrantes estrechos**, sin alterar significativamente el contorno externo del objeto. Es el paso estándar antes de extraer el esqueleto de una huella dactilar.
Ver en [Figura 4.6](#fig-04-sim-ep0406-fechamento) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (líneas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (líneas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, línea por línea.
4. **Datos:** Leer la matriz binaria $f$ (valores $0$ o $1$), línea por línea.
5. **Dilatación:** Calcular $d = f \oplus B$, usando exactamente el algoritmo del EP04_03 (reflejando $B$, sin padding).
6. **Erosión:** Calcular $g = d \ominus B$, usando exactamente el algoritmo del EP04_04 (sin reflejar $B$, sin padding) — ahora aplicado sobre $d$, no sobre $f$.
7. **Salida:** Mostrar la matriz resultante $g$ (el **cierre** de $f$ por $B$) con dimensiones $L \times C$.

#### 📌 Restricciones Computacionales

* **Orden fijo:** Es **siempre** dilatación primero, luego erosión — el orden inverso es la apertura del EP04_05.
* **Mismo $B$:** El elemento estructurante usado en la dilatación y en la erosión debe ser idéntico.
* **Sin padding en ninguna de las dos etapas.**

#### 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Extensividad** | $g \supseteq f$ siempre | El cierre nunca elimina píxeles, solo añade |
| **Idempotencia** | $\text{cierre}(\text{cierre}(f)) = \text{cierre}(f)$ | Aplicarlo de nuevo no cambia nada más |
| **Huecos pequeños** | Menores que $B$ | Se rellenan completamente |
| **Dualidad** | $\text{cierre}(f) = \overline{\text{apertura}(\bar f)}$ | Es la apertura aplicada al "negativo" de la imagen |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros ($0$ o $1$) de la matriz $f$.

**Salida:**

* Matriz resultante en $L$ líneas y $C$ columnas, valores $0$ o $1$.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 8<br>8<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 0 1 1 0 0<br>0 0 1 1 0 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 | 0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0 | Los dos huecos internos no adyacentes se rellenan totalmente |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0406-fechamento" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧩 Simulador EP04_06: Cierre Morfológico</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊕ B) ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en las celdas de <b>f original</b> para encender o apagar píxeles (¡rellena huecos internos!) y ajusta el tamaño del elemento estructurante B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#2c7a7b;">Tamaño de B (Cuadro n×n)</label><br>
      <input type="range" id="sim_ep0406_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#2c7a7b;margin-top:6px;">
      <span id="sim_ep0406_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#2c7a7b;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs d (Dilatação) vs g (Fechamento Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(170px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Clickeable)</span>
        <div id="sim_ep0406_grid_f" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- d = f ⊕ B -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">d = f ⊕ B (Dilatación)</span>
        <div id="sim_ep0406_grid_d" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = d ⊖ B -->
      <div style="background:#fafaf7;border:2px solid #2c7a7b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2c7a7b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = d ⊖ B (Cierre)</span>
        <div id="sim_ep0406_grid_g" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0406_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen (Con Huecos)</button>
      <button id="sim_ep0406_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Limpiar Todo</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0406(root){
    if (!root || root.dataset.simEp0406Init) return;
    root.dataset.simEp0406Init = "1";

    var slN     = root.querySelector('#sim_ep0406_sl_n');
    var vlN     = root.querySelector('#sim_ep0406_vl_n');
    var gF      = root.querySelector('#sim_ep0406_grid_f');
    var gD      = root.querySelector('#sim_ep0406_grid_d');
    var gG      = root.querySelector('#sim_ep0406_grid_g');
    var btnNew  = root.querySelector('#sim_ep0406_btnNew');
    var btnClear= root.querySelector('#sim_ep0406_btnClear');

    var L = 8, C = 8, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 1; y < 7; y++) {
        for (var x = 2; x < 6; x++) f[y][x] = 1;
      }
      f[3][3] = 0;
      f[4][4] = 0;
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#2c7a7b' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#2c7a7b' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var d = dilate(f, B);
      var g = erode(d, B);

      paintEditable(gF, f);
      paintStatic(gD, d);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0406(){
    var root = document.getElementById('sim-ep0406-fechamento');
    if (root) initSimEP0406(root); else setTimeout(tryInitSimEP0406, 200);
  }
  tryInitSimEP0406();
})();
</script>
</div>
""")

**Figura 4.6:** Simulador EP04_06: Cierre Morfológico (g = (f ⊕ B) ⊖ B)


<figure id="fig-04-sim-ep0406-fechamento">
  <img src="imagens/fig-04-sim-ep0406-fechamento.png" alt=" Simulador EP04_06: Cierre Morfológico (g = (f ⊕ B) ⊖ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.6:</strong>  Simulador EP04_06: Cierre Morfológico (g = (f ⊕ B) ⊖ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_06.cpp
// your solution

In [ ]:
TestSuite("EP04_06.cpp").run()

### EP04_07 ⛰️ Dilatación y Erosión con Pesos (mm.dil1 / mm.ero1)

Hasta ahora, el elemento estructurante solo decía "este vecino cuenta" o "no cuenta" — pero en **modelos digitales de elevación** (usados en SIG y en planificación de drenaje urbano), cada vecino debería tener un **peso diferente** dependiendo de la distancia o de la dirección del relieve. Las versiones **ponderadas** de la dilatación y de la erosión, implementadas en `morph.py` como `mm::dil1(f, b)` y `mm::ero1(f, b)`, suman (o restan) el peso de cada vecino antes de tomar el máximo (o mínimo) — generalizando todo lo realizado en los EPs anteriores.
Ver en [Figura 4.7](#fig-04-sim-ep0407-pesos) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $b$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante ponderado.
3. **Pesos:** Leer la matriz $b$ de pesos **enteros** (pueden ser negativos, cero o positivos), fila a fila.
4. **Datos:** Leer la matriz $f$ (la imagen original), fila a fila.
5. **Vecindario sin padding:** Para cada píxel $(y,x)$, recorrer **todas** las posiciones $(by,bx)$ de $b$ (no solo donde valdría $1$ — aquí **todo** peso participa), usando el mismo desplazamiento de los EPs anteriores:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Descartar** todo $(v_y,v_x)$ fuera de $[0,L)\times[0,C)$.
6. **Dilatación ponderada:** Calcular
$$
g_{dil}(y,x) = \max\Big(f(y,x),\ \max_{(v_y,v_x)\ \text{válido}} \big(f(v_y,v_x) + b(by,bx)\big)\Big)
$$
7. **Erosión ponderada:** Calcular, **usando el mismo $b$ y sin reflejar**:
$$
g_{ero}(y,x) = \min\Big(f(y,x),\ \min_{(v_y,v_x)\ \text{válido}} \big(f(v_y,v_x) - b(by,bx)\big)\Big)
$$
8. **Salida:** Mostrar **primero** la matriz $g_{dil}$ completa, y **después** la matriz $g_{ero}$ completa.

#### 📌 Restricciones Computacionales

* **Ninguna de las dos refleja $b$** — la versión ponderada no usa reflexión, incluso en la dilatación (diferente de `mm::dil0`).
* **Todos los pesos participan:** No existe aquí el filtro "$B=1$"; incluso el peso $0$ entra en la cuenta.
* **Sin padding:** los vecinos fuera de la imagen se ignoran, nunca se rellenan virtualmente.
* **Tipo:** La salida puede contener valores negativos o mayores que $255$ — **no** hay *clipping* en este EP.
* **Consejo:** Para eliminar mensajes de desbordamiento al superar los límites del tipo uint8, incluir al inicio del código:
```python
import warnings
warnings.filterwarnings("ignore")
```

#### 🧠 Fundamentación Teórica

| Concepto | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Peso positivo** | "Empuja" el valor del vecino hacia arriba en la dilatación | Simula relieve que asciende en esa dirección |
| **Peso negativo** | Reduce la contribución del vecino | Simula distancia o atenuación direccional |
| **Dualidad ponderada** | $\text{ero1}(f,b) = -\text{dil1}(-f,b)$ | La simetría entre las dos operaciones se mantiene incluso con pesos |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros (pueden ser negativos) de la matriz $b$.
* Siguientes $L$ líneas: elementos enteros de la matriz $f$.

**Salida:**

* Primero la matriz $g_{dil}$ en $L$ líneas y $C$ columnas.
* A continuación la matriz $g_{ero}$ en $L$ líneas y $C$ columnas.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 2 1<br>0 1 0<br>10 20 30<br>40 50 60<br>70 80 90 | 50 60 61<br>80 90 91<br>81 91 92<br>8 9 19<br>9 10 20<br>39 40 50 | Peso central $2$ acelera el crecimiento en la dilatación y la contracción en la erosión |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0407-pesos" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⛰️ Simulador EP04_07: Pesos en el Elemento Estructurante</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">dil1 / ero1</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajuste los pesos del elemento estructurante b con los controles deslizantes y observe el efecto de la dilatación y erosión con pesos sobre la matriz f.</p>

    <!-- Painel dos Pesos b -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Pesos b (Ajuste los Sliders por Celda)</span>
      <div id="sim_ep0407_grid_b" style="display:grid;grid-template-columns:repeat(3, 70px);gap:8px;justify-content:center;user-select:none;"></div>
    </div>

    <!-- Comparativo em 3 Colunas: f original vs dil1 vs ero1 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:14px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original</span>
        <div id="sim_ep0407_grid_f" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- dil1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">dil1(f, b) (Dilatación)</span>
        <div id="sim_ep0407_grid_d" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ero1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">ero1(f, b) (Erosión)</span>
        <div id="sim_ep0407_grid_e" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0407(root){
    if (!root || root.dataset.simEp0407Init) return;
    root.dataset.simEp0407Init = "1";

    var gB = root.querySelector('#sim_ep0407_grid_b');
    var gF = root.querySelector('#sim_ep0407_grid_f');
    var gD = root.querySelector('#sim_ep0407_grid_d');
    var gE = root.querySelector('#sim_ep0407_grid_e');

    var b = [[0, 1, 0], [1, 2, 1], [0, 1, 0]];
    var f = [[10, 20, 30], [40, 50, 60], [70, 80, 90]];
    var L = 3, C = 3;

    function compute() {
      var oy = -3 / 2 + 0.5, ox = -3 / 2 + 0.5;
      var dil = f.map(function(r){ return r.slice(); });
      var ero = f.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < 3; by++) {
            for (var bx = 0; bx < 3; bx++) {
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                var cd = f[vy][vx] + b[by][bx];
                if (cd > dil[y][x]) dil[y][x] = cd;
                var ce = f[vy][vx] - b[by][bx];
                if (ce < ero[y][x]) ero[y][x] = ce;
              }
            }
          }
        }
      }
      return { dil: dil, ero: ero };
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(row, col){
            var wrap = document.createElement('div');
            wrap.style.cssText = 'display:flex;flex-direction:column;align-items:center;background:#fafaf7;border:1px solid #e4dcc8;border-radius:6px;padding:4px;box-sizing:border-box;';

            var val = document.createElement('div');
            val.style.cssText = 'font-family:monospace;font-weight:700;font-size:11px;color:#d35400;margin-bottom:2px;';
            val.textContent = b[row][col];

            var sl = document.createElement('input');
            sl.type = 'range';
            sl.min = '-5';
            sl.max = '5';
            sl.step = '1';
            sl.value = b[row][col];
            sl.style.cssText = 'width:56px;cursor:pointer;accent-color:#d35400;';

            sl.addEventListener('input', function(){
              b[row][col] = parseInt(sl.value, 10);
              val.textContent = b[row][col];
              renderAll();
            });

            wrap.appendChild(val);
            wrap.appendChild(sl);
            gB.appendChild(wrap);
          })(by, bx);
        }
      }
    }

    function paint(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = img[y][x];
          var inten = Math.min(255, Math.max(0, v));
          var fg = inten > 128 ? '#000000' : '#ffffff';
          c.style.cssText = 'width:52px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + inten + ',' + inten + ',' + inten + ');color:' + fg + ';box-sizing:border-box;';
          c.textContent = v;
          grid.appendChild(c);
        }
      }
    }

    function renderAll() {
      var res = compute();
      paint(gF, f);
      paint(gD, res.dil);
      paint(gE, res.ero);
    }

    renderB();
    renderAll();
  }

  function tryInitSimEP0407(){
    var root = document.getElementById('sim-ep0407-pesos');
    if (root) initSimEP0407(root); else setTimeout(tryInitSimEP0407, 200);
  }
  tryInitSimEP0407();
})();
</script>
</div>
""")

**Figura 4.7:** Simulador EP04_07: Dilatación y Erosión con Pesos (mm.dil1 / mm.ero1)


<figure id="fig-04-sim-ep0407-pesos">
  <img src="imagens/fig-04-sim-ep0407-pesos.png" alt=" Simulador EP04_07: Dilatación y Erosión con Pesos (mm.dil1 / mm.ero1) " style="max-width:80%" />
  <figcaption><strong>Figura 4.7:</strong>  Simulador EP04_07: Dilatación y Erosión con Pesos (mm.dil1 / mm.ero1) </figcaption>
</figure>

In [ ]:
%%writefile EP04_07.cpp
// your solution

In [ ]:
TestSuite("EP04_07.cpp").run()

### EP04_08 🌋 Gradiente morfológico, Top-hat y Black-hat

En la **inspección automática de placas de circuito**, tres preguntas aparecen todo el tiempo: ¿dónde están los **bordes** de los componentes? ¿Qué **detalles claros y pequeños** (como puntos de soldadura) se destacan del fondo? ¿Qué **reentrancias oscuras** (como fisuras) esconde el fondo? Un único par erosión/dilatación responde a las tres: el **gradiente morfológico** evidencia contornos, el **top-hat** revela picos estrechos, y el **black-hat** revela valles estrechos — tres herramientas, una sola vecindad.
Ver en [Figura 4.8](#fig-04-sim-ep0408-gradiente) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** Leer los enteros $L$ (filas) y $C$ (columnas) de $f$.
2. **Dimensiones de $B$:** Leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** Leer la matriz $B$ con valores $0$ o $1$, línea a línea.
4. **Datos:** Leer la matriz $f$ (la imagen original, en tonos de gris), línea a línea.
5. **Operadores de base:** Calcular, exactamente como en los EPs 04_03 a 04_06:
   * $d = f \oplus B$ (dilatación),
   * $e = f \ominus B$ (erosión),
   * $\text{abertura} = e \oplus B$,
   * $\text{fechamento} = d \ominus B$.
6. **Gradiente morfológico:** $\text{grad}(y,x) = d(y,x) - e(y,x)$.
7. **Top-hat:** $\text{tophat}(y,x) = f(y,x) - \text{abertura}(y,x)$.
8. **Black-hat:** $\text{blackhat}(y,x) = \text{fechamento}(y,x) - f(y,x)$.
9. **Salida:** Mostrar, **en este orden**, las tres matrices completas: gradiente, top-hat, black-hat.

#### 📌 Restricciones Computacionales

* **Sin padding en ninguna etapa intermedia** — dilatación, erosión, apertura y cierre siguen las mismas reglas de vecindad de los EPs anteriores.
* **No hay *clipping*:** las tres salidas pueden contener cualquier valor entero (el gradiente es siempre $\geq 0$, pero top-hat y black-hat también).
* **Reutilización:** $d$ y $e$ deben calcularse **una única vez** y reutilizarse para montar apertura, cierre y gradiente.

#### 🧠 Fundamentación Teórica

| Operador | Fórmula | Qué revela |
|----------|---------|----------------|
| **Gradiente** | $d - e$ | Bordes: cero en regiones planas, alto en las transiciones |
| **Top-hat** | $f - \text{abertura}(f)$ | Elementos **claros y finos**, más pequeños que $B$ |
| **Black-hat** | $\text{fechamento}(f) - f$ | Elementos **oscuros y finos**, más pequeños que $B$ |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $L_B$.
* Línea 4: Entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros ($0$ o $1$) de la matriz $B$.
* Siguientes $L$ líneas: elementos enteros de la matriz $f$.

**Salida:**

* Matriz gradiente en $L$ filas y $C$ columnas.
* Matriz top-hat en $L$ filas y $C$ columnas.
* Matriz black-hat en $L$ filas y $C$ columnas.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 9<br>9<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 80 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 2 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10 | (gradiente: halo $3\times3=70$ en torno a $(2,2)$ y halo $3\times3=8$ en torno a $(6,6)$, resto $0$)<br>(top-hat: único $70$ en $(2,2)$, resto $0$)<br>(black-hat: único $8$ en $(6,6)$, resto $0$) | Pico aislado se convierte en top-hat; valle aislado se convierte en black-hat; ambos aparecen en el gradiente |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0408-gradiente" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌋 Simulador EP04_08: Gradiente / Top-hat / Black-hat</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">3 operadores, 1 vecindad</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Agregue picos o valles en la matriz f y observe el comportamiento simultáneo de los operadores de gradiente, top-hat y black-hat.</p>

    <!-- Botões de Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0408_btn_pico" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">☀️ Agregar Pico</button>
      <button id="sim_ep0408_btn_vale" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">🕳️ Agregar Valle</button>
      <button id="sim_ep0408_btn_reset" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">↩ Limpiar Todo</button>
    </div>

    <!-- Comparativo em 4 Colunas: f, Gradiente, Top-hat, Black-hat -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(140px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Matriz f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">f (Entrada)</span>
        <div id="sim_ep0408_grid_f" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente -->
      <div style="background:#fafaf7;border:1px solid #8e44ad;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Gradiente</span>
        <div id="sim_ep0408_grid_grad" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Top-hat -->
      <div style="background:#fafaf7;border:1px solid #d35400;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Top-hat</span>
        <div id="sim_ep0408_grid_th" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Black-hat -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Black-hat</span>
        <div id="sim_ep0408_grid_bh" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0408(root){
    if (!root || root.dataset.simEp0408Init) return;
    root.dataset.simEp0408Init = "1";

    var gF   = root.querySelector('#sim_ep0408_grid_f');
    var gGrad= root.querySelector('#sim_ep0408_grid_grad');
    var gTh  = root.querySelector('#sim_ep0408_grid_th');
    var gBh  = root.querySelector('#sim_ep0408_grid_bh');

    var btnPico  = root.querySelector('#sim_ep0408_btn_pico');
    var btnVale  = root.querySelector('#sim_ep0408_btn_vale');
    var btnReset = root.querySelector('#sim_ep0408_btn_reset');

    var L = 9, C = 9, f = [], B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];

    function resetMatrix() {
      f = Array.from({ length: L }, function(){ return new Array(C).fill(10); });
    }

    function morph(img, Bm, mode) {
      var Bref = mode === 'dil' ? Bm.slice().reverse().map(function(r){ return r.slice().reverse(); }) : Bm;
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                if (mode === 'dil' && img[vy][vx] > g[y][x]) g[y][x] = img[vy][vx];
                if (mode === 'ero' && img[vy][vx] < g[y][x]) g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paint(grid, img, cmin, cmax, hue) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var v = img[y][x];
          var t = cmax > cmin ? (v - cmin) / (cmax - cmin) : 0;
          var c = document.createElement('div');
          c.style.cssText = 'width:20px;height:20px;border-radius:3px;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : hue;
          c.style.opacity = v === 0 ? '1' : (0.35 + 0.65 * Math.min(1, t));
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var d = morph(f, B, 'dil');
      var e = morph(f, B, 'ero');
      var ab = morph(e, B, 'dil');
      var fc = morph(d, B, 'ero');

      var grad = f.map(function(r, y){ return r.map(function(_, x){ return d[y][x] - e[y][x]; }); });
      var th   = f.map(function(r, y){ return r.map(function(v, x){ return v - ab[y][x]; }); });
      var bh   = f.map(function(r, y){ return r.map(function(v, x){ return fc[y][x] - v; }); });

      var maxF = Math.max.apply(null, f.map(function(r){ return Math.max.apply(null, r); }));
      var maxG = Math.max.apply(null, grad.map(function(r){ return Math.max.apply(null, r); }));
      var maxTh = Math.max.apply(null, th.map(function(r){ return Math.max.apply(null, r); }));
      var maxBh = Math.max.apply(null, bh.map(function(r){ return Math.max.apply(null, r); }));

      paint(gF, f, 10, maxF || 1, '#7f8c8d');
      paint(gGrad, grad, 0, Math.max(1, maxG), '#8e44ad');
      paint(gTh, th, 0, Math.max(1, maxTh), '#d35400');
      paint(gBh, bh, 0, Math.max(1, maxBh), '#2980b9');
    }

    btnPico.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.min(255, f[y][x] + 60 + Math.floor(Math.random() * 30));
      render();
    });

    btnVale.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.max(0, f[y][x] - 8 - Math.floor(Math.random() * 4));
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMatrix();
      f[2][2] = 80;
      f[6][6] = 2;
      render();
    });

    resetMatrix();
    f[2][2] = 80;
    f[6][6] = 2;
    render();
  }

  function tryInitSimEP0408(){
    var root = document.getElementById('sim-ep0408-gradiente');
    if (root) initSimEP0408(root); else setTimeout(tryInitSimEP0408, 200);
  }
  tryInitSimEP0408();
})();
</script>
</div>
""")

**Figura 4.8:** Simulador EP04_08: Gradiente Morfológico, Top-hat y Black-hat


<figure id="fig-04-sim-ep0408-gradiente">
  <img src="imagens/fig-04-sim-ep0408-gradiente.png" alt=" Simulador EP04_08: Gradiente Morfológico, Top-hat y Black-hat " style="max-width:80%" />
  <figcaption><strong>Figura 4.8:</strong>  Simulador EP04_08: Gradiente Morfológico, Top-hat y Black-hat </figcaption>
</figure>

In [ ]:
%%writefile EP04_08.cpp
// your solution

In [ ]:
TestSuite("EP04_08.cpp").run()

### EP04_09 🗺️ Transformada de Distancia y el "Centro" del Objeto

En **robótica móvil**, al planificar una ruta dentro de un pasillo, el robot quiere saber no solo *dónde* hay espacio libre, sino también **qué tan lejos** está cada punto libre de la pared más cercana. Las rutas más seguras tienden a pasar por el "centro" del pasillo, lejos de los obstáculos.

La **transformada de distancia morfológica** asigna a cada píxel un valor que representa su distancia hasta el borde más cercano, según la métrica definida por el elemento estructurante. Los píxeles cercanos al borde reciben valores bajos, mientras que los píxeles más internos reciben valores mayores. El píxel de valor máximo corresponde a la región más protegida del objeto, frecuentemente asociada a su centro morfológico.

Ver en [Figura 4.9](#fig-04-sim-ep0409-distancia) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** leer los enteros $L$ (filas) y $C$ (columnas) de la imagen $f$.
2. **Dimensiones de $B$:** leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.
3. **Elemento estructurante:** leer la matriz $b$, que contiene el valor $0$ en el centro y valores negativos en las demás posiciones.
4. **Imagen:** leer la matriz binaria $f$ (valores $0$ o $1$), fila a fila.
5. **Preparación:** multiplicar la imagen por $L\times C$, asegurando que los píxeles internos tengan un valor inicial suficientemente alto para la propagación de las distancias.
6. **Transformada de distancia:** calcular la matriz de distancias utilizando el método `mm::dist1(f,b)`.
7. **Salida:** mostrar la matriz resultante de la transformada de distancia.

#### 📌 Restricciones Computacionales

* Utilizar la implementación de erosión ponderada proporcionada por la biblioteca.
* El elemento estructurante puede contener valores negativos arbitrarios.
* La transformada debe obtenerse mediante la aplicación iterativa de erosiones ponderadas hasta alcanzar un punto fijo.

**⚠️ Nota Crucial sobre Lectura de Matrices:** Como el elemento estructurante puede contener valores enteros negativos (por ejemplo, `-1` y `-99`), **no utilice la función `mm::readImg` para leer la matriz $b$**. Esa función convierte los datos al tipo `uint8`, provocando *underflow* y corrompiendo los valores negativos. Lea las $L_B$ filas de $b$ manualmente utilizando el tipo estándar `int`. La imagen $f$ puede seguir leyéndose normalmente con `mm::readImg`.

#### 🧠 Fundamentación Teórica

| Concepto                            | Significado                                                                       | Impacto Visual                               |
| ----------------------------------- | --------------------------------------------------------------------------------- | -------------------------------------------- |
| **$\text{dist}(y,x)$**              | Distancia morfológica hasta el borde más cercano según la métrica definida por $b$ | Los píxeles más internos reciben valores mayores |
| **Valor máximo**                    | Píxel más distante del borde                                                      | Aproxima el centro morfológico del objeto      |
| **Elemento estructurante ponderado** | Define los costos de desplazamiento entre píxeles vecinos                         | Determina la métrica de distancia utilizada   |
| **Objetos finos**                   | Regiones estrechas del objeto                                                       | Producen valores bajos de distancia          |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: entero $L$.
* Línea 2: entero $C$.
* Línea 3: entero $L_B$.
* Línea 4: entero $C_B$.
* Siguientes $L_B$ líneas: elementos enteros de la matriz $b$.
* Siguientes $L$ líneas: elementos binarios ($0$ o $1$) de la matriz $f$.

⚠️ **Nota de implementación:** Los elementos de la matriz $f$ (0 o 1) deben multiplicarse por **255** para generar una imagen binaria adecuada ($0$ y $255$) antes de aplicar la Transformada de Distancia (TD).

**Salida:**

* Matriz de la transformada de distancia en $L$ filas y $C$ columnas.

#### 📌 Ejemplo

| Entrada                                                                                                                                                          | Salida                                                                                                 | Observación                              |
| ---------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- | --------------------------------------- |
| 5<br>9<br>3<br>3<br>-99 -1 -99<br>-1 0 -1<br>-99 -1 -99<br>0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | 0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 2 2 2 2 2 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | Resultado de la transformada de distancia. |

**Nota:** el valor `-99` actúa como una aproximación práctica de $-\infty$, impidiendo la propagación por las diagonales. De esta forma, solo los vecinos horizontal y vertical contribuyen a la distancia, produciendo la distancia de Manhattan.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0409-distancia" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🗺️ Simulador EP04_09: Transformada de Distancia</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Capas de Erosión</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en las celdas para dibujar tu propio objeto o selecciona una forma predefinida para calcular el mapa de distancias en cascada.</p>

    <!-- Grade f Original Clicável -->
    <div style="display:flex;justify-content:center;margin-bottom:14px;">
      <div id="sim_ep0409_grid_f" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

    <!-- Botões de Formas Predefinidas -->
    <div style="text-align:center;margin-bottom:14px;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0409_btn_corredor" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📐 Corredor</button>
      <button id="sim_ep0409_btn_disco" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬤ Disco</button>
      <button id="sim_ep0409_btn_l" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📏 Forma en L</button>
    </div>

    <!-- Título do Mapa de Distâncias -->
    <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Mapa de Distancias Calculado</span>

    <!-- Grade de Distâncias -->
    <div style="display:flex;justify-content:center;">
      <div id="sim_ep0409_grid_dist" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0409(root){
    if (!root || root.dataset.simEp0409Init) return;
    root.dataset.simEp0409Init = "1";

    var gF = root.querySelector('#sim_ep0409_grid_f');
    var gD = root.querySelector('#sim_ep0409_grid_dist');

    var L = 5, C = 9, f = [];
    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];

    function setCorredor() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function setDisco() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var cy = 2, cx = 4;
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (Math.pow(y - cy, 2) + Math.pow((x - cx) * 0.6, 2) <= 4) f[y][x] = 1;
        }
      }
    }

    function setL() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 3; x++) f[y][x] = 1;
      }
      for (var y = 2; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function sameMatrix(a, b) {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (a[y][x] !== b[y][x]) return false;
        }
      }
      return true;
    }

    function render() {
      gF.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:32px;height:32px;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = f[yy][xx] ? '#16a085' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            gF.appendChild(c);
          })(y, x);
        }
      }

      var dist = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var atual = f.map(function(r){ return r.slice(); });
      var nivel = 0;

      while (atual.some(function(r){ return r.some(function(v){ return v === 1; }); })) {
        nivel++;
        for (var y = 0; y < L; y++) {
          for (var x = 0; x < C; x++) {
            if (atual[y][x] === 1) dist[y][x] = nivel;
          }
        }
        var prox = erode(atual, B);
        if (sameMatrix(prox, atual)) break;
        atual = prox;
      }

      gD.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = dist[y][x];
          var t = v / (nivel || 1);
          var inten = Math.round(220 - t * 170);

          c.style.cssText = 'width:32px;height:32px;border-radius:6px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : 'rgb(' + (inten - 60) + ',' + inten + ',' + (inten - 30) + ')';
          c.style.color = v > 0 ? '#ffffff' : '#8a8371';
          c.textContent = v || '';
          gD.appendChild(c);
        }
      }
    }

    root.querySelector('#sim_ep0409_btn_corredor').addEventListener('click', function(){ setCorredor(); render(); });
    root.querySelector('#sim_ep0409_btn_disco').addEventListener('click', function(){ setDisco(); render(); });
    root.querySelector('#sim_ep0409_btn_l').addEventListener('click', function(){ setL(); render(); });

    setCorredor();
    render();
  }

  function tryInitSimEP0409(){
    var root = document.getElementById('sim-ep0409-distancia');
    if (root) initSimEP0409(root); else setTimeout(tryInitSimEP0409, 200);
  }
  tryInitSimEP0409();
})();
</script>
</div>
""")

**Figura 4.9:** Simulador EP04_09: Transformada de Distância (Camadas de Erosión)


<figure id="fig-04-sim-ep0409-distancia">
  <img src="imagens/fig-04-sim-ep0409-distancia.png" alt=" Simulador EP04_09: Transformada de Distância (Camadas de Erosión) " style="max-width:80%" />
  <figcaption><strong>Figura 4.9:</strong>  Simulador EP04_09: Transformada de Distância (Camadas de Erosión) </figcaption>
</figure>

In [ ]:
%%writefile EP04_09.cpp
// your solution

In [ ]:
TestSuite("EP04_09.cpp").run()

### EP04_10 🪙 Separación de *Blobs*, Etiquetado y Descriptores

En una **línea de producción de monedas**, es común que las piezas se toquen entre sí en la cinta transportadora, formando una única mancha conectada en la imagen — un conteo ingenuo erraría el total. La solución clásica combina operaciones morfológicas y análisis de conectividad: primero una **erosión** reduce o rompe conexiones frágiles entre objetos, y luego el **etiquetado de componentes conectados** separa cada objeto en una región distinta. Finalmente, **descriptores geométricos** (área y caja delimitadora) resumen cada componente encontrado.

Ver en [Figura 4.10](#fig-04-sim-ep0410-rotulacao) una simulación de este EP.

#### 📋 Directrices de Implementación

1. **Dimensiones de la imagen:** leer los enteros $L$ (filas) y $C$ (columnas) de $f$.

2. **Dimensiones de $B$:** leer los enteros $L_B$ (filas) y $C_B$ (columnas) del elemento estructurante.

3. **Elemento estructurante:** leer la matriz $B$, que contiene valores $0$ o $1$, fila a fila.

4. **Datos:** leer la matriz binaria $f$ (valores $0$ o $1$), fila a fila.

5. **Separación:** calcular
   $$
   f_{ero} = f \ominus B
   $$
   usando erosión binaria plana (como en el EP04_04), eliminando conexiones frágiles entre objetos.

6. **Etiquetado:** sobre $f_{ero}$, identificar componentes conectados usando conectividad definida por la vecindad $B$. El etiquetado debe seguir un barrido *raster*: al encontrar un píxel $1$ aún no etiquetado, asignar una nueva etiqueta entera creciente a partir de 1 y propagar esa etiqueta a toda la región conectada.

7. **Descriptores:** para cada etiqueta $k$, calcular:

   * **Área:** número de píxeles pertenecientes a la etiqueta;
   * **Caja delimitadora:** $$(y_{min}, x_{min}, y_{max}, x_{max})$$

8. **Salida:** mostrar el número total de etiquetas y, a continuación, una línea por etiqueta en el formato:
   $$
   k,\ \text{área},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
   $$

#### 📌 Restricciones Computacionales

* La erosión debe aplicarse antes del etiquetado.
* La conectividad es fija y está definida por la vecindad anterior.
* El elemento estructurante $B$ no interfiere en la conectividad del etiquetado.
* Sin *padding* en ninguna etapa.
* El orden de las etiquetas sigue la primera detección en el barrido *raster*.

#### 🧠 Fundamentación Teórica

| Concepto           | Significado                                   | Impacto                                              |
| ------------------ | --------------------------------------------- | ---------------------------------------------------- |
| Puente fino        | Conexión estrecha entre objetos               | Puede ser eliminado por la erosión morfológica       |
| Conectividad       | Definida por el conjunto $$\mathcal{N}(y,x)$$ | Determina qué píxeles pertenecen al mismo componente |
| Área               | Número de píxeles por componente              | Estimación directa del tamaño del objeto             |
| Caja delimitadora  | Extensión espacial de la etiqueta             | Resumen geométrico del componente                    |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: entero $L$
* Línea 2: entero $C$
* Línea 3: entero $L_B$
* Línea 4: entero $C_B$
* Siguientes $L_B$ líneas: matriz $B$
* Siguientes $L$ líneas: matriz $f$

**Salida:**

* Línea 1: número total de etiquetas encontradas
* Líneas siguientes:
  $$
  k,\ \text{área},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
  $$

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0410-rotulacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪙 Simulador EP04_10: Monedas Pegadas → Separadas → Contadas</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">erosión + etiqueta + descriptores</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajusta el grosor del puente entre las monedas y observa cómo la erosión morfológica separa los objetos para el conteo y la extracción de descriptores (área y cuadro delimitador).</p>

    <!-- Controle de Espessura da Ponte -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#b9770e;">Grosor del Puente entre las Monedas</label><br>
      <input type="range" id="sim_ep0410_sl_p" min="1" max="3" step="1" value="1" style="width:60%;cursor:pointer;accent-color:#b9770e;margin-top:6px;">
      <span id="sim_ep0410_vl_p" style="font-family:monospace;font-size:12px;font-weight:700;color:#b9770e;margin-left:8px;">1 píxel</span>
    </div>

    <!-- Comparativo Lado a Lado: f original vs Rótulos Pós-Erosão -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original (ligadas) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Pegadas)</span>
        <div id="sim_ep0410_grid_f" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Após Erosão + Rótulos -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Tras Erosión + Etiquetas</span>
        <div id="sim_ep0410_grid_lab" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Painel Informativo / Descritores -->
    <div id="sim_ep0410_info" style="background:#fef5e7;border:1px solid #f8c471;border-radius:8px;padding:10px 14px;font-size:11px;color:#7d5a00;text-align:center;line-height:1.5;"></div>

  </div>
</div>

<script>
(function(){
  function initSimEP0410(root){
    if (!root || root.dataset.simEp0410Init) return;
    root.dataset.simEp0410Init = "1";

    var slP  = root.querySelector('#sim_ep0410_sl_p');
    var vlP  = root.querySelector('#sim_ep0410_vl_p');
    var gF   = root.querySelector('#sim_ep0410_grid_f');
    var gL   = root.querySelector('#sim_ep0410_grid_lab');
    var info = root.querySelector('#sim_ep0410_info');

    var L = 7, C = 10;
    var B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
    var palette = ['#e74c3c', '#27ae60', '#2980b9', '#8e44ad', '#d35400'];

    function buildF(p) {
      var f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 6; y++) {
        for (var x = 1; x < 4; x++) f[y][x] = 1;
      }
      for (var y = 1; y < 6; y++) {
        for (var x = 6; x < 9; x++) f[y][x] = 1;
      }
      var midRow = 3;
      for (var dy = 0; dy < p; dy++) {
        var ry = midRow - Math.floor(p / 2) + dy;
        for (var x = 4; x < 6; x++) f[ry][x] = 1;
      }
      return f;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function labelK8(img) {
      var labels = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var dirs = [[-1, -1], [-1, 0], [-1, 1], [0, -1], [0, 1], [1, -1], [1, 0], [1, 1]];
      var cur = 0, desc = [];

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (img[y][x] === 1 && labels[y][x] === 0) {
            cur++;
            var stack = [[y, x]];
            labels[y][x] = cur;
            var area = 0, miny = y, maxy = y, minx = x, maxx = x;

            while (stack.length) {
              var cell = stack.pop();
              var cy = cell[0], cx = cell[1];
              area++;
              if (cy < miny) miny = cy;
              if (cy > maxy) maxy = cy;
              if (cx < minx) minx = cx;
              if (cx > maxx) maxx = cx;

              dirs.forEach(function(d){
                var ny = cy + d[0], nx = cx + d[1];
                if (ny >= 0 && ny < L && nx >= 0 && nx < C && img[ny][nx] === 1 && labels[ny][nx] === 0) {
                  labels[ny][nx] = cur;
                  stack.push([ny, nx]);
                }
              });
            }
            desc.push({k: cur, area: area, miny: miny, minx: minx, maxy: maxy, maxx: maxx});
          }
        }
      }
      return {labels: labels, desc: desc};
    }

    function paintBin(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] ? '#b9770e' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintLabels(grid, labels) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var k = labels[y][x];
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;color:#ffffff;box-sizing:border-box;';
          c.style.background = k > 0 ? palette[(k - 1) % palette.length] : '#fafaf7';
          c.textContent = k > 0 ? k : '';
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var p = parseInt(slP.value, 10) || 1;
      vlP.textContent = p + ' px';
      var f = buildF(p);
      var fe = erode(f, B);
      var res = labelK8(fe);

      paintBin(gF, f);
      paintLabels(gL, res.labels);

      var txt = '<b>' + res.desc.length + ' objeto(s) detectado(s) após a erosão.</b><br>';
      res.desc.forEach(function(d){
        txt += 'Rótulo ' + d.k + ': área = ' + d.area + ', bbox = (' + d.miny + ',' + d.minx + ') → (' + d.maxy + ',' + d.maxx + ')<br>';
      });
      if (res.desc.length < 2) {
        txt += '<i>A ponte ainda é espessa demais para a erosão 3×3 — as moedas continuam fundidas em 1 só objeto.</i>';
      }
      info.innerHTML = txt;
    }

    slP.addEventListener('input', render);

    render();
  }

  function tryInitSimEP0410(){
    var root = document.getElementById('sim-ep0410-rotulacao');
    if (root) initSimEP0410(root); else setTimeout(tryInitSimEP0410, 200);
  }
  tryInitSimEP0410();
})();
</script>
</div>
""")

**Figura 4.10:** Simulador EP04_10: Separación de Blobs, Etiquetado y Descriptores


<figure id="fig-04-sim-ep0410-rotulacao">
  <img src="imagens/fig-04-sim-ep0410-rotulacao.png" alt=" Simulador EP04_10: Separación de Blobs, Etiquetado y Descriptores " style="max-width:80%" />
  <figcaption><strong>Figura 4.10:</strong>  Simulador EP04_10: Separación de Blobs, Etiquetado y Descriptores </figcaption>
</figure>

In [ ]:
%%writefile EP04_10.cpp
// your solution

In [ ]:
TestSuite("EP04_10.cpp").run()